# 🎯 Final Project: End-to-End Agent with Evals

---

## 🌟 Project Overview

By the end of this notebook, you'll build:

1. A complete **Research Agent** with tools
2. **Tracing** enabled with LangSmith
3. **Evaluation dataset** with test cases
4. **Multiple evaluators** for comprehensive testing
5. **Eval pipeline** that runs and reports

---

## ⏱️ Time Estimate
**~45 minutes**

In [ ]:
!pip install -q langchain langchain-openai langgraph langsmith
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI Key: ")

# Set up tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
if "LANGCHAIN_API_KEY" in os.environ:
    pass  # Already set

print("✅ Setup complete!")

## 📦 Step 1: Create the Agent

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# Define tools
@tool
def search_web(query: str) -> str:
    """Search the web for information.
    
    Args:
        query: What to search for
    
    Returns:
        Search results
    """
    return f"[Search results for '{query}'] Found relevant articles."

@tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression.
    
    Args:
        expression: Math expression (e.g., '2+2', '15*23')
    
    Returns:
        Result
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_weather(location: str) -> str:
    """Get weather for a location.
    
    Args:
        location: City name
    
    Returns:
        Weather info
    """
    return f"Weather in {location}: Sunny, 72°F"

tools = [search_web, calculate, get_weather]

# Create agent
llm = ChatOpenAI(model="gpt-4o-mini")
agent = create_react_agent(llm, tools)

print("✅ Research Agent Created!")
print(f"   Tools: {[t.name for t in tools]}")

## 📊 Step 2: Create Test Dataset

In [ ]:
test_dataset = [
    # Math queries
    {"query": "What is 25 * 17?", "expected_tool": "calculate", "expected": "425"},
    {"query": "What is 100 + 200?", "expected_tool": "calculate", "expected": "300"},
    
    # Weather queries
    {"query": "What's the weather in Tokyo?", "expected_tool": "get_weather", "expected": "Tokyo"},
    {"query": "Is it sunny in Paris?", "expected_tool": "get_weather", "expected": "Paris"},
    
    # Search queries
    {"query": "Find info about AI agents", "expected_tool": "search_web", "expected": "AI agents"},
    {"query": "What is Python?", "expected_tool": "search_web", "expected": "Python"},
    
    # General (no tool needed)
    {"query": "Hello!", "expected_tool": None, "expected": "Hello"},
]

print(f"✅ Created dataset with {len(test_dataset)} test cases")
for case in test_dataset[:3]:
    print(f"   • {case['query']}")

## 🔍 Step 3: Build Evaluators

In [ ]:
def eval_output(reference: str, prediction: str) -> dict:
    """Check if output contains expected answer."""
    correct = reference.lower() in prediction.lower()
    return {
        "passed": correct,
        "score": 100 if correct else 0,
        "feedback": "Correct" if correct else "Incorrect"
    }

def eval_tool_selection(expected_tool: str, actual_tools: list) -> dict:
    """Check if correct tool was used."""
    if expected_tool is None:
        # No tool needed
        passed = len(actual_tools) == 0
    else:
        passed = expected_tool in actual_tools
    return {
        "passed": passed,
        "score": 100 if passed else 0,
        "feedback": f"Expected: {expected_tool}, Got: {actual_tools}"
    }

def eval_hallucination(reference: str, prediction: str) -> dict:
    """Check for hallucination markers."""
    markers = ["i'm not sure", "might be", "probably", "perhaps"]
    has_marker = any(m.lower() in prediction.lower() for m in markers)
    # If there's an expected reference, should match
    if reference:
        passes = not has_marker and reference.lower() in prediction.lower()
    else:
        passes = not has_marker  # Fine if no reference
    return {
        "passed": passes,
        "score": 100 if passes else 0,
        "feedback": "Hallucination" if has_marker else "OK"
    }

## ▶️ Step 4: Run Evaluation Pipeline

In [ ]:
def run_agent(query: str) -> tuple:
    """Run agent and return prediction + traces."""
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": query}]})
        response = result["messages"][-1].content
        
        # Extract tool calls (simplified)
        tool_calls = []
        for msg in result["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                tool_calls.extend([tc["name"] for tc in msg.tool_calls])
        
        return response, tool_calls
    except Exception as e:
        return f"Error: {e}", []

def run_evaluation():
    """Run complete evaluation."""
    results = []
    
    for case in test_dataset:
        query = case["query"]
        expected = case["expected"]
        expected_tool = case.get("expected_tool")
        
        # Run agent
        prediction, tool_calls = run_agent(query)
        
        # Run evaluators
        output_result = eval_output(expected, prediction)
        tool_result = eval_tool_selection(expected_tool, tool_calls)
        hall_result = eval_hallucination(expected, prediction)
        
        results.append({
            "query": query,
            "prediction": prediction,
            "tools_used": tool_calls,
            "evaluations": {
                "output": output_result,
                "tool": tool_result,
                "hallucination": hall_result
            },
            "all_passed": output_result["passed"] and tool_result["passed"]
        })
    
    return results

print("🔄 Running Evaluation...")
eval_results = run_evaluation()
print("✅ Evaluation Complete!")

## 📊 Step 5: View Results

In [ ]:
print("=" * 70)
print("📊 EVALUATION RESULTS")
print("=" * 70)

total_passed = 0
for result in eval_results:
    status = "✅" if result["all_passed"] else "❌"
    print(f"{status} Query: {result['query'][:40]}...")
    print(f"   Prediction: {result['prediction'][:50]}...")
    print(f"   Tools: {result['tools_used']}")
    print()
    if result["all_passed"]:
        total_passed += 1

# Summary
passed_output = sum(1 for r in eval_results if r["evaluations"]["output"]["passed"])
passed_tool = sum(1 for r in eval_results if r["evaluations"]["tool"]["passed"])
passed_hall = sum(1 for r in eval_results if r["evaluations"]["hallucination"]["passed"])

total = len(eval_results)
print("=" * 70)
print("📈 SUMMARY")
print("=" * 70)
print(f"Output Correct: {passed_output}/{total} ({100*passed_output/total:.0f}%)")
print(f"Tool Correct:  {passed_tool}/{total} ({100*passed_tool/total:.0f}%)")
print(f"No Hallucination: {passed_hall}/{total} ({100*passed_hall/total:.0f}%)")
print(f"\n✅ ALL PASSED: {total_passed}/{total} ({100*total_passed/total:.0f}%)")

## 🔧 Step 6: Improve Based on Results

In [ ]:
print("📝 IMPROVEMENTS TO MAKE")
print("=" * 60)

failures = [r for r in eval_results if not r["all_passed"]]
if failures:
    print(f"⚠️ {len(failures)} test cases failed. Consider:")
    
    for f in failures:
        print(f"   Query: {f['query']}")
        if not f["evaluations"]["output"]["passed"]:
            print(f"   → Improve output matching")
        if not f["evaluations"]["tool"]["passed"]:
            print(f"   → Improve tool descriptions / routing")
        if not f["evaluations"]["hallucination"]["passed"]:
            print(f"   → Add grounding / RAG")
else:
    print("✅ All tests passed! Ready for production.")

## 🎉 What You've Built!

```
┌─────────────────────────────────────────────────────────────┐
│              COMPLETE AGENT EVAL SYSTEM                    │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ✅ Agent with Tools (search, calculate, weather)         │
│  ✅ LangGraph Tracing Enabled                           │
│  ✅ Test Dataset (7 test cases)                        │
│  ✅ Multiple Evaluators (output, tool, hallucination)    │
│  ✅ Evaluation Pipeline (run & report)                 │
│  ✅ Results Analysis + Improvement Plan              │
│                                                             │
│  Ready for production with more test cases!            │
└─────────────────────────────────────────────────────────────┘
```

## 🚀 Next Steps to Extend

1. **Add more test cases** - 50+ for confidence
2. **Use LangSmith** - Full trace visualization
3. **LLM-as-Judge** - More sophisticated evaluation
4. **RAG eval** - Check citation accuracy
5. **Multi-turn** - Test conversation memory
6. **Deploy** - Put into production monitoring

## 🎓 What You Learned

1. **LLM Basics** - How LLMs work, tokens, prompts
2. **Agent Components** - Tools, memory, state, routing
3. **Frameworks** - LangChain, LangGraph
4. **Observability** - LangSmith, tracing
5. **Agent Evals** - Types, datasets, evaluators
6. **Error Handling** - Common bugs + fixes

**Congratulations! 🎉 You've completed the Agent Evals Tutorial!**

---

## 📚 Continue Learning

• LangSmith Docs: https://docs.smith.langchain.com
• LangChain Docs: https://python.langchain.com
• LangGraph Docs: https://langchain-ai.github.io/langgraph/

---

*Created as part of Agent Evals Tutorial*